In [36]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

from sklearn.metrics import classification_report, confusion_matrix, mean_absolute_error

In [37]:
clf_df = pd.read_csv("data/classification_escalation.csv")
reg_df = pd.read_csv("data/resolution_time_regression.csv")

display(clf_df.head())
display(reg_df.head())

,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,escalated
0,T0000,2.0,15.0,Medium,Email,61.818896,0
1,T0001,9.0,6.0,Critical,Chat,110.024317,0
2,T0002,10.0,8.0,High,Phone,36.311244,0
3,T0003,2.0,1.0,Medium,Phone,85.127855,0
4,T0004,8.0,14.0,Medium,Chat,18.225059,0


,ticket_id,complexity_score,team_experience_years,priority,channel,customer_tenure_months,resolution_time_hours
0,R0000,7.0,10.0,Medium,Chat,32.638377,8.38724
1,R0001,4.0,15.0,Medium,Chat,45.273760,0.50000
2,R0002,4.0,13.0,Low,Portal,44.878323,0.50000
3,R0003,1.0,10.0,High,Phone,40.095034,0.50000
4,R0004,3.0,15.0,Low,Portal,47.451134,0.50000


In [38]:
print("Classification dataset shape:", clf_df.shape)
print("Regression dataset shape:", reg_df.shape)

print("\nClassification dataset information:")
clf_df.info()

print("\nRegression dataset information:")
reg_df.info()

Classification dataset shape: (900, 7)
Regression dataset shape: (800, 7)

Classification dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 900 entries, 0 to 899
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id               900 non-null    str    
 1   complexity_score        878 non-null    float64
 2   team_experience_years   864 non-null    float64
 3   priority                887 non-null    str    
 4   channel                 891 non-null    str    
 5   customer_tenure_months  869 non-null    float64
 6   escalated               900 non-null    int64  
dtypes: float64(3), int64(1), str(3)
memory usage: 49.3 KB

Regression dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ticket_id              

In [39]:
print("Classification missing values:")
display(clf_df.isnull().sum())

print("\nRegression missing values:")
display(reg_df.isnull().sum())

Classification missing values:


ticket_id                  0
complexity_score          22
team_experience_years     36
priority                  13
channel                    9
customer_tenure_months    31
escalated                  0
dtype: int64


Regression missing values:


ticket_id                  0
complexity_score          16
team_experience_years     24
priority                   0
channel                    0
customer_tenure_months    20
resolution_time_hours      0
dtype: int64

In [40]:
print("Classification target distribution:")
display(clf_df["escalated"].value_counts())

print("\nClassification target percentages:")
display(clf_df["escalated"].value_counts(normalize=True) * 100)

Classification target distribution:


escalated
0    709
1    191
Name: count, dtype: int64


Classification target percentages:


escalated
0    78.777778
1    21.222222
Name: proportion, dtype: float64

In [41]:
# experiment A

X_clf = clf_df.drop(columns=["ticket_id", "escalated"])
y_clf = clf_df["escalated"]

print("Features:")
display(X_clf.head())

print("Target:")
display(y_clf.head())

Features:


,complexity_score,team_experience_years,priority,channel,customer_tenure_months
0,2.0,15.0,Medium,Email,61.818896
1,9.0,6.0,Critical,Chat,110.024317
2,10.0,8.0,High,Phone,36.311244
3,2.0,1.0,Medium,Phone,85.127855
4,8.0,14.0,Medium,Chat,18.225059


Target:


0    0
1    0
2    0
3    0
4    0
Name: escalated, dtype: int64

In [42]:
# Split the data into training and testing sets
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf,
    y_clf,
    test_size=0.2,
    stratify=y_clf,
    random_state=42
)

print("Training set size:", X_train_clf.shape)
print("Testing set size:", X_test_clf.shape)

Training set size: (720, 5)
Testing set size: (180, 5)


In [43]:
# Identify numerical and categorical columns(preprocess)
numeric_features_clf = X_train_clf.select_dtypes(include=["number"]).columns.tolist()

categorical_features_clf = X_train_clf.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:", numeric_features_clf)
print("Categorical features:", categorical_features_clf)

Numerical features: ['complexity_score', 'team_experience_years', 'customer_tenure_months']
Categorical features: ['priority', 'channel']


C:\Users\hp\AppData\Local\Temp\ipykernel_13124\2818486877.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features_clf = X_train_clf.select_dtypes(include=["object", "category"]).columns.tolist()


In [44]:
# Build the preprocessing pipelines

numeric_transformer_clf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])


In [45]:
categorical_transformer_clf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [46]:
# tell to use applies the numerical pipeline to numerical columns.
# tell to use the categorical pipeline to categorical columns.
preprocessor_clf = ColumnTransformer(transformers=[
    ("num", numeric_transformer_clf, numeric_features_clf),
    ("cat", categorical_transformer_clf, categorical_features_clf)
])

if we preprocess the entire dataset before splitting or before cross-validation, information from the validation or test data could influence the preprocessing process. This is called data leakage.

For example, if we calculate the mean or median using all the data, the preprocessing step has already seen information from data that should have been kept separate.

By placing preprocessing inside a pipeline, the preprocessing steps are fitted using only the training portion of each CV fold, and then applied to the validation portion.

This helps prevent data leakage and gives us a more reliable evaluation.

now

Build and compare classification models

we will compare three machine learning models to see which one performs better at predicting whether a ticket will be escalated.

In [47]:
# Create the 5-fold Stratified Cross-Validation strategy

cv_clf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [48]:
# Build the Logistic Regression model

logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_clf),
    ("model", LogisticRegression(max_iter=1000))
])

In [49]:
# Build the KNN model
# K-Nearest Neighbors (KNN) predicts a class by looking at the nearest data points to a new observation.

knn_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor_clf),
    ("model", KNeighborsClassifier())
])


In [50]:
# Build the Decision Tree model
# A Decision Tree makes predictions by asking a series of questions about the data.

tree_pipeline_clf = Pipeline(steps=[
    ("preprocessor", preprocessor_clf),
    ("model", DecisionTreeClassifier(random_state=42))
])

In [51]:
# Compare the three models using Cross-Validation

models_clf = {
    "Logistic Regression": logistic_pipeline,
    "KNN": knn_pipeline,
    "Decision Tree": tree_pipeline_clf
}

clf_results = []

for name, model in models_clf.items():
    scores = cross_validate(
        model,
        X_train_clf,
        y_train_clf,
        cv=cv_clf,
        scoring="f1",
        return_train_score=False
    )


    clf_results.append({
        "Model": name,
        "Mean F1": scores["test_score"].mean(),
        "Std F1": scores["test_score"].std()
    })

clf_comparison = pd.DataFrame(clf_results)

display(clf_comparison.sort_values(by="Mean F1", ascending=False))

,Model,Mean F1,Std F1
0,Logistic Regression,0.392651,0.062747
1,KNN,0.360835,0.050295
2,Decision Tree,0.330666,0.058126


Mean F1 => tells us the average F1 score across the 5 validation folds.

Std F1 => It tells us how much the F1 scores varied across the 5 folds. A lower standard deviation means the scores were more consistent across those folds.

Logistic Regression: Highest mean F1 (0.392651) among the three models we tested. So it performed best on average across the 5 folds.


In [52]:

# 1. Define the hyperparameter search space
param_grid_lr = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

# 2. Create GridSearchCV
grid_search_lr = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=param_grid_lr,
    scoring="f1",
    cv=cv_clf,
    n_jobs=-1
)

# 3. Run the hyperparameter search
grid_search_lr.fit(X_train_clf, y_train_clf)

# 4. Display the best parameter and score
print("Best parameters:", grid_search_lr.best_params_)
print("Best CV F1-score:", grid_search_lr.best_score_)

# 5. Display the results for all tested C values
lr_tuning_results = pd.DataFrame(grid_search_lr.cv_results_)

display(
    lr_tuning_results[
        ["param_model__C", "mean_test_score", "std_test_score"]
    ].sort_values(by="mean_test_score", ascending=False)
)


Best parameters: {'model__C': 10}
Best CV F1-score: 0.4225932078257659


,param_model__C,mean_test_score,std_test_score
4,100.00,0.422593,0.088225
3,10.00,0.422593,0.088225
2,1.00,0.392651,0.062747
1,0.10,0.327873,0.046086
0,0.01,0.000000,0.000000


In [53]:
# 1. Get the best tuned model
best_clf_model = grid_search_lr.best_estimator_

# 2. Predict on the test set
y_pred_clf = best_clf_model.predict(X_test_clf)

# 3. Display classification report
print("Classification Report:")
print(classification_report(y_test_clf, y_pred_clf))

# 4. Display confusion matrix
cm_clf = confusion_matrix(y_test_clf, y_pred_clf)

print("Confusion Matrix:")
print(cm_clf)

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.96      0.89       142
           1       0.62      0.21      0.31        38

    accuracy                           0.81       180
   macro avg       0.72      0.59      0.60       180
weighted avg       0.78      0.81      0.77       180

Confusion Matrix:
[[137   5]
 [ 30   8]]



Logistic Regression was selected for tuning because it had the highest mean F1-score (0.392651) among the three models in the initial comparison. I used GridSearchCV to test five values of C: 0.01, 0.1, 1, 10, and 100, using 5-fold Stratified Cross-Validation. The best C value reported was 10, with a mean CV F1-score of 0.422593. C = 100 achieved the same score.

After tuning, the model was evaluated on the test set. It achieved 81% accuracy. For class 0 (not escalated), the model performed well, with 96% recall. However, for class 1 (escalated), it achieved only 21% recall, meaning it missed many tickets that actually needed escalation.

The confusion matrix showed that the model correctly predicted 137 tickets as not escalated and 8 tickets as escalated. It incorrectly predicted 5 tickets as escalated, while it missed 30 tickets that were actually escalated.

Although the model performed well at identifying tickets that were not escalated, it was less effective at identifying escalated tickets. Since identifying escalated tickets is important, the model's low recall for class 1 is a limitation that should be considered.